<a href="https://colab.research.google.com/github/Hassanmufezshaikh/AI-Agents/blob/main/multiAgent_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade google-adk google-genai

  Using cached google_genai-2.4.0-py3-none-any.whl.metadata (52 kB)


In [ ]:
import google.adk
print(google.adk.__version__)

1.34.0


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
print(" Tunnel Components imported successfully")



 Tunnel Components imported successfully


In [ ]:
import os
from google.colab import userdata
userdata.get('gemeni')

try:
  GOOGLE_API_KEY = userdata.get('gemeni')
  os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
  print("Gemini API Key Setup Complete")
except Exception as e :
  print("Authencation Error: Please add 'GEMENI_API_KEY' to your kaggale secrets, Details : {e}")

Gemini API Key Setup Complete


In [ ]:
from google.adk.agents import (
    Agent,
    SequentialAgent,
    ParallelAgent,
    LoopAgent
)

from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, FunctionTool
from google.genai import types

print("ADK components imported successfully.")

ADK components imported successfully.


In [ ]:
import google.adk.tools as tools

print(dir(tools))

['APIHubToolset', 'AgentTool', 'AgentTool', 'Any', 'ApiRegistry', 'AuthToolArguments', 'BaseTool', 'DiscoveryEngineSearchTool', 'ExampleTool', 'FunctionTool', 'LongRunningFunctionTool', 'MCPToolset', 'McpToolset', 'SearchResultMode', 'TYPE_CHECKING', 'ToolContext', 'TransferToAgentTool', 'VertexAiSearchTool', '_LAZY_MAPPING', '__all__', '__builtins__', '__cached__', '__dir__', '__doc__', '__file__', '__getattr__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_automatic_function_calling_util', '_forwarding_artifact_service', '_function_parameter_parse_util', '_function_tool_declarations', 'agent_tool', 'base_tool', 'base_toolset', 'computer_use', 'enterprise_web_search', 'exit_loop', 'function_tool', 'get_user_choice', 'google_maps_grounding', 'google_search', 'google_search', 'google_search_tool', 'importlib', 'load_artifacts', 'load_memory', 'logging', 'preload_memory', 'set_model_response_tool', 'sys', 'tool_configs', 'tool_confirmation', 'tool_context', 'transfe

In [ ]:
from google.genai import types

retry_config=types.HttpRetryOptions(
attempts=5, # Maximum retry attempts
exp_base=7, # Delay multiplier
initial_delay=1, # Initial delay before first retry (in seconds)
http_status_codes=[429, 500, 503, 504]
)

In [ ]:
# Research Agent: Its job is to use the google_search tool and present findings.

research_agent = Agent(
    name="ResearchAgent",

    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),

    instruction="""
    You are a specialized research agent.
    Your only job is to use the Google Search tool
    to find 2-3 pieces of relevant information
    on the given topic and present the findings with citations.
    """,

    tools=[google_search],

    # The result of this agent will be stored in session state
    output_key="research_findings",
)

print("research_agent created.")

research_agent created.


In [ ]:
#Summarizer Agent: Its job is to summarize the text it receives.
summarizer_agent = Agent(
name="SummarizerAgent",
model=Gemini(
model="gemini-2.5-flash-lite",
retry_options=retry_config
),
# The instruction is modified to request a bulleted list for a clear output format.
instruction="""Read the provided research findings: {research_findings},
create a concise summary as a bulleted list with 3-5 key points.
""",
output_key="final_summary"
)
print(" summarizer_agent created.")

 summarizer_agent created.


In [ ]:
#Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
root_agent = Agent (
name="ResearchCoordinator",
model=Gemini(
model="gemini-2.5-flash-lite",
retry_options=retry_config
),
#This instruction tells the root agent HOW to use its tools (which are the other agents).
instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the ResearchAgent tool to find relevant information on the topic provided by the user.
2. Next, after receiving the research findings, you MUST call the 'SummarizerAgent tool to create a concise summary.
3. Finally, present the final summary clearly to the user as your response.""",
)

#We wrap the sub-agents in AgentTool to make them callable tools for the root agent.
tools=[AgentTool(research_agent), AgentTool(summarizer_agent)],
print(" root_agent created.")

 root_agent created.


In [ ]:
runner = InMemoryRunner(agent=root_agent)
print("InMemoryRunner created.")
response =  await runner.run_debug(
    "What are the Latest Advancements in quantum computing and what they do mean for AI?"
)

InMemoryRunner created.

 ### Created new session: debug_session_id

User > What are the Latest Advancements in quantum computing and what they do mean for AI?
ResearchCoordinator > I will start by researching the latest advancements in quantum computing.
Then, I will research the implications of these advancements for artificial intelligence.
Finally, I will summarize the gathered information on the latest advancements in quantum computing and their implications for AI to answer your question.
1. Call ResearchAgent tool with the query: "latest advancements in quantum computing"
2. Call ResearchAgent tool with the query: "implications of quantum computing advancements for AI"
3. Call SummarizerAgent tool with the research findings from steps 1 and 2.
4. Present the summarized response to the user.
